# 🧠 LLM Training Pipeline from Scratch

This notebook implements a complete pipeline to:
1. **Build and train a small GPT-style LLM from scratch** (~110M params, fits in 4GB RAM)
2. **Evaluate** the base model with standard NLP metrics
3. **Extend with reasoning capabilities** using multiple approaches:
   - Chain-of-Thought (CoT) Supervised Fine-Tuning
   - Self-Consistency Decoding
   - Process Reward Model (PRM) guided search
   - RLHF-style reasoning with PPO (simplified)
   - STaR (Self-Taught Reasoner) bootstrapping

---
**Requirements:**
```bash
pip install torch transformers datasets tokenizers accelerate tqdm numpy
```
**Hardware:** CPU-friendly (~110M params, ~440MB weights in fp32)

---
## Part 1 — Base LLM: Architecture, Training & Evaluation

### 1.1 Imports & Configuration

In [ ]:
import math
import time
import copy
import json
import random
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# ── Model config (~110M params, comfortably under 4 GB RAM) ──────────────────
@dataclass
class GPTConfig:
    vocab_size   : int   = 50257   # GPT-2 BPE vocab
    max_seq_len  : int   = 512
    n_layers     : int   = 12
    n_heads      : int   = 12
    d_model      : int   = 768
    d_ff         : int   = 3072    # 4 * d_model
    dropout      : float = 0.1
    bias         : bool  = True

cfg = GPTConfig()

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params    : {total:,}')
    print(f'  Trainable params: {train:,}')
    print(f'  Approx RAM (fp32): {total * 4 / 1e9:.2f} GB')
    return total

### 1.2 Model Architecture — GPT-style Decoder-only Transformer

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalisation (faster than LayerNorm)."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * (x / rms)


class CausalSelfAttention(nn.Module):
    """
    Multi-head causal (autoregressive) self-attention.
    Uses Flash-Attention if available (torch >= 2.0), otherwise manual.
    """
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads  = cfg.n_heads
        self.d_head   = cfg.d_model // cfg.n_heads
        self.d_model  = cfg.d_model

        self.qkv_proj = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model,     bias=cfg.bias)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)

        # Causal mask (lower-triangular)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(cfg.max_seq_len, cfg.max_seq_len)).view(
                1, 1, cfg.max_seq_len, cfg.max_seq_len)
        )
        self.use_flash = hasattr(F, 'scaled_dot_product_attention')

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(self.d_model, dim=-1)

        # Reshape to (B, H, T, D_head)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        if self.use_flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.attn_drop.p if self.training else 0.0,
                is_causal=True
            )
        else:
            scale = 1.0 / math.sqrt(self.d_head)
            att = (q @ k.transpose(-2, -1)) * scale
            att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_drop(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(y))


class FeedForward(nn.Module):
    """Position-wise FFN with GELU activation (same as GPT-2)."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block (more stable than post-norm)."""
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = CausalSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ff    = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class NanoGPT(nn.Module):
    """
    Decoder-only transformer — GPT-2 small equivalent (~117M params).
    Architecture: token embedding + positional embedding → N transformer
    blocks → RMSNorm → linear head (weight-tied to embeddings).
    """
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict({
            'tok_emb' : nn.Embedding(cfg.vocab_size, cfg.d_model),
            'pos_emb' : nn.Embedding(cfg.max_seq_len, cfg.d_model),
            'drop'    : nn.Dropout(cfg.dropout),
            'blocks'  : nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)]),
            'norm'    : RMSNorm(cfg.d_model),
        })
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Weight tying: lm_head shares weights with token embedding
        self.transformer['tok_emb'].weight = self.lm_head.weight

        # Init weights
        self.apply(self._init_weights)
        # Scale residual projections (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith('out_proj.weight') or name.endswith('net.2.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.max_seq_len, f'Sequence length {T} > max {self.cfg.max_seq_len}'

        pos = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        tok = self.transformer['tok_emb'](idx)
        pos = self.transformer['pos_emb'](pos)
        x = self.transformer['drop'](tok + pos)

        for block in self.transformer['blocks']:
            x = block(x)
        x = self.transformer['norm'](x)
        logits = self.lm_head(x)  # (B, T, V)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-100
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self, idx, max_new_tokens: int = 100,
        temperature: float = 1.0, top_k: int = 50,
        top_p: float = 0.9, stop_token: Optional[int] = None
    ):
        """Autoregressive generation with top-k + top-p (nucleus) sampling."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            # Top-k filtering
            if top_k > 0:
                topk_vals = torch.topk(logits, top_k).values
                logits[logits < topk_vals[:, -1:]] = float('-inf')

            # Top-p (nucleus) filtering
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_logits[cumprobs - F.softmax(sorted_logits, dim=-1) > top_p] = float('-inf')
                logits.scatter_(1, sorted_idx, sorted_logits)

            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_tok], dim=1)

            if stop_token is not None and next_tok.item() == stop_token:
                break
        return idx


# ── Instantiate & count params ────────────────────────────────────────────────
model = NanoGPT(cfg).to(DEVICE)
print('NanoGPT parameter count:')
count_params(model)

### 1.3 Tokeniser (BPE via Hugging Face `tokenizers`)

In [ ]:
# We use the pretrained GPT-2 tokenizer (50257-token BPE vocab) for convenience.
# To train a custom BPE tokenizer from scratch on your own corpus, see the
# `train_custom_tokenizer` helper at the bottom of this cell.

from transformers import GPT2TokenizerFast

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default
PAD_ID = tokenizer.pad_token_id
EOS_ID = tokenizer.eos_token_id

print(f'Vocab size : {tokenizer.vocab_size}')
print(f'EOS token  : "{tokenizer.eos_token}" (id {EOS_ID})')

# ── Helper to encode / decode ─────────────────────────────────────────────────
def encode(text: str, max_length: int = 512) -> torch.Tensor:
    ids = tokenizer.encode(text, truncation=True, max_length=max_length)
    return torch.tensor(ids, dtype=torch.long)

def decode(ids) -> str:
    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()
    return tokenizer.decode(ids, skip_special_tokens=True)


# ── Optional: Train a custom BPE tokenizer from scratch ───────────────────────
def train_custom_tokenizer(corpus_texts: List[str], vocab_size: int = 8000, save_path: str = 'tokenizer'):
    """
    Trains a BPE tokenizer from scratch on a list of text strings.
    Requires: pip install tokenizers
    """
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tok = Tokenizer(BPE(unk_token='[UNK]'))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'],
        min_frequency=2
    )
    tok.train_from_iterator(corpus_texts, trainer=trainer)
    tok.save(f'{save_path}/tokenizer.json')
    print(f'Custom tokenizer saved → {save_path}/tokenizer.json')
    return tok

### 1.4 Dataset & DataLoader

In [ ]:
# ── Tiny in-notebook corpus for demonstration ─────────────────────────────────
# In practice, replace this with a streaming HuggingFace dataset loader,
# e.g.: from datasets import load_dataset; ds = load_dataset('openwebtext', split='train', streaming=True)

DEMO_CORPUS = [
    "The quick brown fox jumps over the lazy dog.",
    "Language models learn to predict the next token in a sequence.",
    "Transformer architectures use self-attention to model long-range dependencies.",
    "Training a large language model requires vast amounts of text data.",
    "Backpropagation computes gradients by applying the chain rule.",
    "The Adam optimizer adapts the learning rate for each parameter.",
    "Tokenization splits text into subword units for model processing.",
    "Positional encodings inject sequence order information into embeddings.",
    "Residual connections and layer normalization stabilize deep network training.",
    "Fine-tuning adapts a pretrained model to a downstream task.",
] * 500   # replicate to get a non-trivial number of batches


class TextDataset(Dataset):
    """
    Tokenises a list of texts and creates fixed-length overlapping windows
    for causal language modelling (next-token prediction).
    """
    def __init__(self, texts: List[str], tokenizer, seq_len: int = 128, stride: int = 64):
        self.seq_len = seq_len
        self.examples: List[torch.Tensor] = []

        all_ids: List[int] = []
        for text in texts:
            ids = tokenizer.encode(text)
            all_ids.extend(ids)
            all_ids.append(EOS_ID)

        all_ids = torch.tensor(all_ids, dtype=torch.long)
        for start in range(0, len(all_ids) - seq_len, stride):
            chunk = all_ids[start : start + seq_len + 1]  # +1 for targets
            if len(chunk) == seq_len + 1:
                self.examples.append(chunk)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        chunk = self.examples[idx]
        x = chunk[:-1]  # input tokens
        y = chunk[1:]   # target tokens (shifted by 1)
        return x, y


SEQ_LEN    = 128
BATCH_SIZE = 8

# 90 / 10 train / val split
n_train = int(len(DEMO_CORPUS) * 0.9)
train_ds = TextDataset(DEMO_CORPUS[:n_train],  tokenizer, seq_len=SEQ_LEN)
val_ds   = TextDataset(DEMO_CORPUS[n_train:], tokenizer, seq_len=SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train examples: {len(train_ds):,}')
print(f'Val examples  : {len(val_ds):,}')
print(f'Train batches : {len(train_loader):,}')

### 1.5 Training Loop

In [ ]:
@dataclass
class TrainConfig:
    epochs           : int   = 3
    lr               : float = 3e-4
    weight_decay     : float = 0.1
    grad_clip        : float = 1.0
    warmup_steps     : int   = 100
    eval_interval    : int   = 200   # evaluate every N steps
    log_interval     : int   = 50
    save_best        : bool  = True
    checkpoint_path  : str   = 'best_model.pt'


def get_lr(step: int, cfg: TrainConfig, total_steps: int) -> float:
    """Linear warmup → cosine decay."""
    if step < cfg.warmup_steps:
        return cfg.lr * step / max(1, cfg.warmup_steps)
    progress = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return cfg.lr * 0.5 * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, device, max_batches: int = 50) -> float:
    model.eval()
    total_loss, n = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(n, 1)


def train(
    model: NanoGPT,
    train_loader: DataLoader,
    val_loader: DataLoader,
    tcfg: TrainConfig = TrainConfig()
) -> Dict:
    """
    Full training loop.
    Returns history dict with train_loss, val_loss, val_ppl per eval step.
    """
    history = {'step': [], 'train_loss': [], 'val_loss': [], 'val_ppl': []}

    # Separate weight decay for 2-D tensors (weight matrices) vs 1-D (biases/norms)
    decay_params    = [p for n, p in model.named_parameters() if p.dim() >= 2 and p.requires_grad]
    no_decay_params = [p for n, p in model.named_parameters() if p.dim() <  2 and p.requires_grad]
    optimizer = AdamW(
        [{'params': decay_params,    'weight_decay': tcfg.weight_decay},
         {'params': no_decay_params, 'weight_decay': 0.0}],
        lr=tcfg.lr, betas=(0.9, 0.95), eps=1e-8
    )

    total_steps = tcfg.epochs * len(train_loader)
    best_val    = float('inf')
    step        = 0
    t0          = time.time()

    for epoch in range(1, tcfg.epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            # Learning rate schedule
            lr = get_lr(step, tcfg, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            _, loss = model(x, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
            optimizer.step()

            running_loss += loss.item()
            step += 1

            # ── Logging ──────────────────────────────────────────────────────
            if step % tcfg.log_interval == 0:
                avg = running_loss / tcfg.log_interval
                elapsed = time.time() - t0
                print(f'  Epoch {epoch} | Step {step:5d}/{total_steps} | '
                      f'loss {avg:.4f} | lr {lr:.2e} | {elapsed:.1f}s')
                running_loss = 0.0

            # ── Evaluation ───────────────────────────────────────────────────
            if step % tcfg.eval_interval == 0:
                val_loss = evaluate(model, val_loader, DEVICE)
                val_ppl  = math.exp(min(val_loss, 20))  # cap to avoid overflow
                print(f'  >>> Val loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}')
                history['step'].append(step)
                history['train_loss'].append(loss.item())
                history['val_loss'].append(val_loss)
                history['val_ppl'].append(val_ppl)

                if tcfg.save_best and val_loss < best_val:
                    best_val = val_loss
                    torch.save(model.state_dict(), tcfg.checkpoint_path)
                    print(f'  ✓ Saved new best model → {tcfg.checkpoint_path}')

    print(f'\nTraining complete in {time.time()-t0:.1f}s | Best val loss: {best_val:.4f}')
    return history


tcfg = TrainConfig(epochs=3, lr=3e-4, eval_interval=200, log_interval=50)
print('Starting training …')
history = train(model, train_loader, val_loader, tcfg)

### 1.6 Training Curve Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

if history['step']:
    fig = plt.figure(figsize=(12, 4))
    gs  = gridspec.GridSpec(1, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(history['step'], history['train_loss'], label='Train loss', color='#3B82F6')
    ax1.plot(history['step'], history['val_loss'],   label='Val loss',   color='#F59E0B')
    ax1.set_title('Loss Curves', fontweight='bold')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Cross-entropy loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    ax2.plot(history['step'], history['val_ppl'], color='#10B981')
    ax2.set_title('Validation Perplexity', fontweight='bold')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Perplexity')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()
    print('Plot saved → training_curves.png')
else:
    print('No eval checkpoints recorded — run more steps or decrease eval_interval.')

### 1.7 Evaluation Suite

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluation metrics implemented from scratch (no external eval libraries):
#   1. Perplexity  — standard LM metric
#   2. Bits-per-character (BPC)
#   3. Top-k accuracy
#   4. BLEU score (corpus-level, pure Python)
#   5. Qualitative generation samples
# ─────────────────────────────────────────────────────────────────────────────

def compute_perplexity(model, loader, device, n_batches: int = 100) -> float:
    """Computes average per-token perplexity over n_batches."""
    model.eval()
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches:
                break
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            log_probs = F.log_softmax(logits, dim=-1)
            nll = F.nll_loss(log_probs.view(-1, log_probs.size(-1)), y.view(-1), reduction='sum')
            total_nll    += nll.item()
            total_tokens += y.numel()
    return math.exp(total_nll / total_tokens)


def compute_bpc(model, loader, device, n_batches: int = 100) -> float:
    """Bits-per-character = log2(perplexity) — useful for character-level comparison."""
    ppl = compute_perplexity(model, loader, device, n_batches)
    return math.log2(ppl)


def compute_topk_accuracy(model, loader, device, k: int = 5, n_batches: int = 50) -> float:
    """Fraction of targets that appear in the model's top-k predictions."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches:
                break
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            topk = torch.topk(logits, k, dim=-1).indices   # (B, T, k)
            target_exp = y.unsqueeze(-1).expand_as(topk)
            correct += (topk == target_exp).any(dim=-1).sum().item()
            total   += y.numel()
    return correct / max(total, 1)


def _ngram_counts(tokens, n):
    counts = defaultdict(int)
    for i in range(len(tokens) - n + 1):
        counts[tuple(tokens[i:i+n])] += 1
    return counts

def bleu_score(references: List[List[int]], hypotheses: List[List[int]], max_n: int = 4) -> float:
    """Corpus-level BLEU score (pure Python, no sacrebleu)."""
    clipped_counts = defaultdict(int)
    total_counts   = defaultdict(int)
    ref_len, hyp_len = 0, 0

    for ref, hyp in zip(references, hypotheses):
        ref_len += len(ref)
        hyp_len += len(hyp)
        for n in range(1, max_n + 1):
            ref_c = _ngram_counts(ref, n)
            hyp_c = _ngram_counts(hyp, n)
            for ng, cnt in hyp_c.items():
                clipped_counts[n] += min(cnt, ref_c.get(ng, 0))
                total_counts[n]   += cnt

    log_bleu = 0.0
    for n in range(1, max_n + 1):
        if total_counts[n] == 0:
            return 0.0
        p = clipped_counts[n] / total_counts[n]
        log_bleu += (1/max_n) * math.log(max(p, 1e-9))

    bp = min(1.0, math.exp(1 - ref_len / max(hyp_len, 1)))
    return bp * math.exp(log_bleu)


# ── Run all evals ─────────────────────────────────────────────────────────────
print('═' * 60)
print('EVALUATION RESULTS')
print('═' * 60)

ppl  = compute_perplexity(model, val_loader, DEVICE)
bpc  = compute_bpc(model, val_loader, DEVICE)
acc5 = compute_topk_accuracy(model, val_loader, DEVICE, k=5)
acc1 = compute_topk_accuracy(model, val_loader, DEVICE, k=1)

print(f'Perplexity (PPL)   : {ppl:.2f}')
print(f'Bits-per-char (BPC): {bpc:.4f}')
print(f'Top-1 accuracy     : {acc1*100:.2f}%')
print(f'Top-5 accuracy     : {acc5*100:.2f}%')

# ── BLEU on short continuation task ──────────────────────────────────────────
# Feed first half of each sentence, generate the rest, compare to ground truth
test_texts = DEMO_CORPUS[n_train:n_train+20]
refs, hyps = [], []
model.eval()
for text in test_texts:
    ids = encode(text)
    half = max(4, len(ids)//2)
    prefix = ids[:half].unsqueeze(0).to(DEVICE)
    generated = model.generate(prefix, max_new_tokens=half, temperature=0.7, top_k=40)
    refs.append(ids[half:].tolist())
    hyps.append(generated[0, half:].tolist())

bleu = bleu_score(refs, hyps)
print(f'BLEU-4 (continuation): {bleu:.4f}')
print('═' * 60)

# ── Qualitative generation samples ───────────────────────────────────────────
print('\nGeneration samples:')
prompts = [
    'Transformer architectures use',
    'The Adam optimizer',
    'Backpropagation computes',
]
model.eval()
for prompt in prompts:
    inp = encode(prompt).unsqueeze(0).to(DEVICE)
    out = model.generate(inp, max_new_tokens=30, temperature=0.8, top_k=50)
    print(f'  Prompt: "{prompt}"')
    print(f'  Output: "{decode(out[0])}"')
    print()

---
## Part 2 — Reasoning Capabilities

We extend the base LLM with reasoning via **5 complementary approaches**:

| # | Method | Core Idea |
|---|--------|-----------|
| 1 | Chain-of-Thought SFT | Supervised fine-tuning on `<think>…</think>` traces |
| 2 | Self-Consistency | Sample K answers, majority-vote the final answer |
| 3 | Process Reward Model (PRM) | Score each reasoning step; use beam search guided by step scores |
| 4 | PPO-style RLHF | Policy gradient with a verifier reward signal |
| 5 | STaR (Self-Taught Reasoner) | Bootstrap: generate rationales, keep correct ones, fine-tune, repeat |

### 2.1 Approach 1 — Chain-of-Thought Supervised Fine-Tuning (CoT SFT)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CoT SFT: fine-tune the model on (question → <think> step-by-step </think> answer)
# The model learns to produce explicit reasoning chains before final answers.
# Format: "Q: {question}\nA: <think>\n{rationale}\n</think>\n{answer}"
# ─────────────────────────────────────────────────────────────────────────────

# ── Synthetic CoT dataset (arithmetic reasoning) ──────────────────────────────
def make_cot_examples(n: int = 200) -> List[str]:
    """Generates simple arithmetic CoT examples."""
    examples = []
    ops = ['+', '-', '*']
    for _ in range(n):
        a, b = random.randint(1, 20), random.randint(1, 20)
        op   = random.choice(ops)
        ans  = eval(f'{a}{op}{b}')
        if op == '+':
            rationale = f'We need to add {a} and {b}. Starting from {a}, we count up {b} more steps. {a} + {b} = {ans}.'
        elif op == '-':
            rationale = f'We need to subtract {b} from {a}. Starting from {a}, we count down {b} steps. {a} - {b} = {ans}.'
        else:
            rationale = f'We need to multiply {a} by {b}. This means adding {a} a total of {b} times. {a} × {b} = {ans}.'
        text = f'Q: What is {a} {op} {b}?\nA: <think>\n{rationale}\n</think>\n{ans}'
        examples.append(text)
    return examples


class CoTDataset(Dataset):
    """
    SFT dataset for CoT fine-tuning.
    Loss is only computed on the answer portion (after </think>),
    achieved by masking the rationale tokens with -100.
    """
    def __init__(self, examples: List[str], tokenizer, seq_len: int = 128, loss_on_answer_only: bool = True):
        self.items = []
        close_tag  = tokenizer.encode('</think>')[0]   # we'll mask up to here

        for text in examples:
            ids = tokenizer.encode(text, truncation=True, max_length=seq_len + 1)
            if len(ids) < 4:
                continue
            ids = torch.tensor(ids, dtype=torch.long)
            x   = ids[:-1]
            y   = ids[1:].clone()

            if loss_on_answer_only:
                # Mask everything before (and including) the </think> token
                mask_until = len(y)
                for pos, tok in enumerate(y):
                    if tok.item() == close_tag:
                        mask_until = pos + 1
                        break
                y[:mask_until] = -100

            self.items.append((x, y))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

    @staticmethod
    def collate_fn(batch):
        xs, ys = zip(*batch)
        max_len = max(x.size(0) for x in xs)
        X = torch.zeros(len(xs), max_len, dtype=torch.long)
        Y = torch.full((len(xs), max_len), -100, dtype=torch.long)
        for i, (x, y) in enumerate(zip(xs, ys)):
            X[i, :x.size(0)] = x
            Y[i, :y.size(0)] = y
        return X, Y


cot_examples = make_cot_examples(400)
cot_ds  = CoTDataset(cot_examples[:320], tokenizer)
cot_val = CoTDataset(cot_examples[320:], tokenizer)
cot_loader = DataLoader(cot_ds, batch_size=8, shuffle=True, collate_fn=CoTDataset.collate_fn)

print(f'CoT SFT training examples: {len(cot_ds)}')
print('\nExample CoT training text:')
print(cot_examples[0])


# ── Fine-tune ─────────────────────────────────────────────────────────────────
cot_model = copy.deepcopy(model)   # start from pretrained base
cot_optimizer = AdamW(cot_model.parameters(), lr=1e-4, weight_decay=0.01)

print('\nFine-tuning with CoT SFT …')
for epoch in range(1, 4):
    cot_model.train()
    total_loss = 0.0
    for x, y in cot_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        cot_optimizer.zero_grad()
        _, loss = cot_model(x, y)
        loss.backward()
        nn.utils.clip_grad_norm_(cot_model.parameters(), 1.0)
        cot_optimizer.step()
        total_loss += loss.item()
    print(f'  Epoch {epoch} | avg loss {total_loss/len(cot_loader):.4f}')


# ── Inference with CoT ────────────────────────────────────────────────────────
def cot_inference(model, question: str, max_new: int = 80) -> str:
    prompt = f'Q: {question}\nA: <think>\n'
    ids    = encode(prompt).unsqueeze(0).to(DEVICE)
    out    = model.generate(ids, max_new_tokens=max_new, temperature=0.7, top_k=40)
    return decode(out[0])

print('\nCoT inference examples:')
for q in ['What is 7 + 5?', 'What is 12 - 4?', 'What is 3 * 6?']:
    print(f'  Q: {q}')
    print(f'  A: {cot_inference(cot_model, q)[:200]}')
    print()

### 2.2 Approach 2 — Self-Consistency Decoding

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Self-Consistency (Wang et al., 2022):
#   1. Sample K independent reasoning chains at temperature > 0
#   2. Extract the final answer from each chain
#   3. Return the majority-voted answer
# Crucially, this requires NO additional training — it's a decoding strategy.
# ─────────────────────────────────────────────────────────────────────────────

import re

def extract_final_answer(text: str) -> str:
    """
    Extracts the answer after </think>.
    Falls back to the last numeric token if tag not found.
    """
    if '</think>' in text:
        answer_part = text.split('</think>')[-1].strip()
    else:
        answer_part = text.strip()
    # Extract first integer / decimal number
    numbers = re.findall(r'-?\d+(?:\.\d+)?', answer_part)
    return numbers[0] if numbers else answer_part[:30]


def self_consistency(
    model,
    question: str,
    K: int = 5,
    temperature: float = 0.9,
    max_new_tokens: int = 80
) -> Tuple[str, Dict]:
    """
    Run K independent chain-of-thought samples, then majority-vote.
    Returns (majority_answer, vote_counts).
    """
    prompt = f'Q: {question}\nA: <think>\n'
    ids    = encode(prompt).unsqueeze(0).to(DEVICE)

    vote_counts: Dict[str, int] = defaultdict(int)
    chains = []

    for k in range(K):
        out  = model.generate(ids, max_new_tokens=max_new_tokens,
                               temperature=temperature, top_k=50)
        text = decode(out[0])
        ans  = extract_final_answer(text)
        vote_counts[ans] += 1
        chains.append((text, ans))

    majority_ans = max(vote_counts, key=vote_counts.get)
    return majority_ans, dict(vote_counts)


print('Self-Consistency evaluation:')
for question in ['What is 8 + 9?', 'What is 15 - 7?']:
    greedy_ans = extract_final_answer(cot_inference(cot_model, question, max_new=80))
    sc_ans, votes = self_consistency(cot_model, question, K=5)
    print(f'  Q: {question}')
    print(f'    Greedy answer      : {greedy_ans}')
    print(f'    Self-consistent ans: {sc_ans}  (votes: {votes})')
    print()

### 2.3 Approach 3 — Process Reward Model (PRM)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Process Reward Model (Lightman et al., 2023 — "Let's Verify Step by Step"):
#
#  • A separate small model (PRM) is trained to assign a correctness score
#    to EACH reasoning step (not just the final answer).
#  • During inference, beam search is guided by the PRM step scores.
#
# Architecture: shared transformer backbone, linear head outputs scalar score.
# ─────────────────────────────────────────────────────────────────────────────

class ProcessRewardModel(nn.Module):
    """
    Lightweight PRM: small transformer backbone + a regression head.
    Input: tokenised (question + partial reasoning chain)
    Output: scalar ∈ [0, 1] — estimated probability the step is correct
    """
    def __init__(self, vocab_size: int = 50257, d_model: int = 256,
                 n_heads: int = 4, n_layers: int = 4, max_len: int = 256):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4, dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        self.max_len = max_len

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        B, T = ids.shape
        T = min(T, self.max_len)
        ids = ids[:, :T]
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x   = self.tok_emb(ids) + self.pos_emb(pos)
        x   = self.encoder(x)          # (B, T, d_model)
        cls = x[:, -1, :]              # last token as pooling
        return self.head(cls).squeeze(-1)   # (B,)


prm = ProcessRewardModel().to(DEVICE)
print('PRM parameter count:')
count_params(prm)


# ── Synthetic PRM training data ───────────────────────────────────────────────
# Label 1.0 = correct step, 0.0 = incorrect/incomplete step
def make_prm_data(n: int = 300) -> Tuple[List[torch.Tensor], List[float]]:
    texts, labels = [], []
    for _ in range(n):
        a, b = random.randint(1,20), random.randint(1,20)
        correct_ans = a + b
        wrong_ans   = correct_ans + random.choice([-2,-1,1,2])

        # Correct step
        texts.append(f'Q: What is {a} + {b}?\nStep: {a} + {b} = {correct_ans}')
        labels.append(1.0)
        # Incorrect step
        texts.append(f'Q: What is {a} + {b}?\nStep: {a} + {b} = {wrong_ans}')
        labels.append(0.0)
    return texts, labels


class PRMDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.items = []
        for t, l in zip(texts, labels):
            ids = tokenizer.encode(t, truncation=True, max_length=max_len)
            self.items.append((torch.tensor(ids, dtype=torch.long), torch.tensor(l, dtype=torch.float)))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

    @staticmethod
    def collate(batch):
        xs, ys = zip(*batch)
        max_len = max(x.size(0) for x in xs)
        X = torch.zeros(len(xs), max_len, dtype=torch.long)
        for i, x in enumerate(xs):
            X[i, :x.size(0)] = x
        return X, torch.stack(ys)


prm_texts, prm_labels = make_prm_data(400)
prm_ds     = PRMDataset(prm_texts[:320], prm_labels[:320], tokenizer)
prm_loader = DataLoader(prm_ds, batch_size=16, shuffle=True, collate_fn=PRMDataset.collate)

prm_opt = AdamW(prm.parameters(), lr=1e-3)
prm_criterion = nn.BCELoss()

print('\nTraining Process Reward Model …')
for epoch in range(1, 6):
    prm.train()
    total = 0.0
    for x, y in prm_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        prm_opt.zero_grad()
        preds = prm(x)
        loss  = prm_criterion(preds, y)
        loss.backward()
        prm_opt.step()
        total += loss.item()
    print(f'  Epoch {epoch} | BCE loss {total/len(prm_loader):.4f}')


# ── PRM-guided beam search ────────────────────────────────────────────────────
@torch.no_grad()
def prm_guided_generate(
    lm: NanoGPT, prm: ProcessRewardModel,
    question: str,
    beam_width: int = 3,
    max_new_tokens: int = 60,
    temperature: float = 0.8
) -> str:
    """
    Beam search where each beam is scored by the PRM at each step.
    Selects the beam with highest cumulative PRM score.
    """
    prompt = f'Q: {question}\nA: <think>\n'
    init_ids = encode(prompt).unsqueeze(0).to(DEVICE)

    # Each beam: (token_ids_tensor, cumulative_prm_score)
    beams = [(init_ids, 0.0)]

    for _ in range(max_new_tokens):
        candidates = []
        for ids, score in beams:
            logits, _ = lm(ids[:, -lm.cfg.max_seq_len:])
            logits = logits[:, -1, :] / temperature
            top_ids = torch.topk(logits, beam_width).indices[0]  # (beam_width,)

            for tok in top_ids:
                new_ids = torch.cat([ids, tok.view(1,1)], dim=1)
                prm_score = prm(new_ids[:, -128:]).item()
                candidates.append((new_ids, score + prm_score))

        # Keep top beam_width beams by cumulative PRM score
        beams = sorted(candidates, key=lambda c: c[1], reverse=True)[:beam_width]

    best_ids, _ = beams[0]
    return decode(best_ids[0])


print('\nPRM-guided generation:')
for q in ['What is 6 + 7?', 'What is 9 - 3?']:
    print(f'  Q: {q}')
    out = prm_guided_generate(cot_model, prm, q, beam_width=3)
    print(f'  A: {out[:180]}')
    print()

### 2.4 Approach 4 — RLHF-style Reasoning with PPO (Simplified)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Simplified PPO for Reasoning (GRPO-inspired, Sheng et al. 2024):
#
# Pipeline:
#  1. Policy LM generates K solutions for each question.
#  2. A verifier (outcome reward model) scores each solution: +1 if correct, 0 if not.
#  3. Advantage A_i = r_i - mean(r_1..K)   (group normalisation)
#  4. PPO-clip loss updates the policy to increase log-prob of high-advantage tokens.
#  5. KL divergence from reference model constrains how far the policy drifts.
#
# Note: This is a pedagogically clear implementation, not production-scale.
# ─────────────────────────────────────────────────────────────────────────────

def verifier_reward(question: str, answer_text: str) -> float:
    """
    Simple outcome verifier for arithmetic:
    +1.0 if the extracted number equals the correct answer, else 0.0.
    """
    # Parse question: 'What is A op B?'
    m = re.search(r'(\d+)\s*([+\-*])\s*(\d+)', question)
    if not m:
        return 0.0
    a, op, b = int(m.group(1)), m.group(2), int(m.group(3))
    correct = int(eval(f'{a}{op}{b}'))
    pred    = extract_final_answer(answer_text)
    try:
        return 1.0 if int(float(pred)) == correct else 0.0
    except (ValueError, TypeError):
        return 0.0


def compute_log_probs(model, ids: torch.Tensor) -> torch.Tensor:
    """Returns per-token log-probabilities for a given sequence."""
    with torch.no_grad():
        logits, _ = model(ids)
    log_probs = F.log_softmax(logits, dim=-1)  # (B, T, V)
    # Gather log-prob of the actual tokens (shift: predict token t+1 from t)
    token_log_probs = log_probs[:, :-1, :].gather(
        -1, ids[:, 1:].unsqueeze(-1)
    ).squeeze(-1)  # (B, T-1)
    return token_log_probs


def ppo_reasoning_step(
    policy:    NanoGPT,
    ref_model: NanoGPT,
    optimizer: torch.optim.Optimizer,
    questions: List[str],
    K: int   = 4,
    clip_eps: float = 0.2,
    kl_coeff: float = 0.05,
    temperature: float = 0.9,
    max_new: int = 60
) -> Dict[str, float]:
    """
    One PPO update step over a mini-batch of questions.
    Returns dict of metrics.
    """
    policy.train()
    ref_model.eval()

    total_loss, total_reward, n_samples = 0.0, 0.0, 0

    for q in questions:
        prompt = f'Q: {q}\nA: <think>\n'
        prompt_ids = encode(prompt).unsqueeze(0).to(DEVICE)

        # ── Step 1: Sample K rollouts ────────────────────────────────────────
        rollouts, rewards = [], []
        with torch.no_grad():
            for _ in range(K):
                out  = policy.generate(prompt_ids, max_new_tokens=max_new,
                                       temperature=temperature, top_k=40)
                text = decode(out[0])
                r    = verifier_reward(q, text)
                rollouts.append(out)
                rewards.append(r)

        # ── Step 2: Group-normalised advantages ──────────────────────────────
        r_tensor   = torch.tensor(rewards, dtype=torch.float)
        advantages = (r_tensor - r_tensor.mean()) / (r_tensor.std() + 1e-8)
        total_reward += r_tensor.mean().item()

        # ── Step 3: PPO-clip loss for each rollout ───────────────────────────
        for ids, adv in zip(rollouts, advantages.tolist()):
            # Only backprop on the generated portion (after prompt)
            prompt_len = prompt_ids.size(1)
            if ids.size(1) <= prompt_len + 1:
                continue

            gen_ids = ids                          # (1, T_full)

            # Policy log-probs (requires grad)
            logits_pol, _ = policy(gen_ids)
            lp_pol = F.log_softmax(logits_pol, dim=-1)
            lp_pol = lp_pol[:, prompt_len-1:-1, :].gather(
                -1, gen_ids[:, prompt_len:].unsqueeze(-1)
            ).squeeze(-1).sum(-1)                  # scalar log-prob sum

            # Reference log-probs (no grad)
            with torch.no_grad():
                logits_ref, _ = ref_model(gen_ids)
                lp_ref = F.log_softmax(logits_ref, dim=-1)
                lp_ref = lp_ref[:, prompt_len-1:-1, :].gather(
                    -1, gen_ids[:, prompt_len:].unsqueeze(-1)
                ).squeeze(-1).sum(-1)

            # Ratio r_t = exp(log π_θ - log π_ref)  (PPO uses old_policy, we approximate with ref)
            log_ratio = lp_pol - lp_ref
            ratio     = log_ratio.exp()

            adv_t = torch.tensor(adv, device=DEVICE)
            # Clipped surrogate objective
            surr1 = ratio * adv_t
            surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_t
            policy_loss = -torch.min(surr1, surr2)

            # KL penalty: keep policy close to reference
            kl_penalty = kl_coeff * log_ratio

            loss = (policy_loss + kl_penalty).mean()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            n_samples  += 1

    return {
        'ppo_loss'  : total_loss / max(n_samples, 1),
        'avg_reward': total_reward / len(questions)
    }


# ── Run PPO reasoning training ────────────────────────────────────────────────
rl_model   = copy.deepcopy(cot_model)     # policy initialised from CoT SFT model
ref_model  = copy.deepcopy(cot_model)     # frozen reference
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

rl_optimizer = AdamW(rl_model.parameters(), lr=5e-6, weight_decay=0.0)

# Construct arithmetic questions
train_questions = [f'What is {random.randint(1,20)} + {random.randint(1,20)}?' for _ in range(60)]

print('PPO Reasoning Training:')
for ppo_step in range(1, 6):
    batch_q = random.sample(train_questions, 8)
    metrics = ppo_reasoning_step(rl_model, ref_model, rl_optimizer, batch_q, K=4)
    print(f'  Step {ppo_step} | PPO loss {metrics["ppo_loss"]:.4f} | avg reward {metrics["avg_reward"]:.3f}')

print('\nRL-tuned model generation samples:')
for q in ['What is 5 + 8?', 'What is 14 - 6?']:
    ans = extract_final_answer(cot_inference(rl_model, q))
    print(f'  Q: {q}  →  predicted answer: {ans}')

### 2.5 Approach 5 — STaR: Self-Taught Reasoner Bootstrapping

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STaR (Zelikman et al., 2022):
#
# Iterative bootstrapping loop:
#   1. For each training problem, generate K rationale attempts.
#   2. Keep only those where the final answer is CORRECT (verified).
#   3. For problems where all K attempts are WRONG, perform "hint" rationalisation:
#      re-generate the rationale given the correct answer as a hint.
#   4. Fine-tune the model on the (problem, kept_rationale) dataset.
#   5. Repeat for N rounds — model continually improves its own training data.
# ─────────────────────────────────────────────────────────────────────────────

def star_generate_rationale(
    model, question: str,
    K: int = 4, temperature: float = 0.9,
    max_new: int = 70
) -> Optional[str]:
    """
    Generate K rationales; return the first correct one, or None.
    """
    prompt = f'Q: {question}\nA: <think>\n'
    ids    = encode(prompt).unsqueeze(0).to(DEVICE)
    for _ in range(K):
        out  = model.generate(ids, max_new_tokens=max_new,
                               temperature=temperature, top_k=40)
        text = decode(out[0])
        if verifier_reward(question, text) == 1.0:
            return text
    return None


def star_hint_rationalise(
    model, question: str, correct_answer: int,
    max_new: int = 60
) -> str:
    """
    Provide the correct answer as a hint and ask the model to generate
    a rationale that arrives at it (rationalisation step in STaR).
    """
    prompt = (f'Q: {question}\n'
              f'Hint: The correct answer is {correct_answer}.\n'
              f'A: <think>\n')
    ids = encode(prompt).unsqueeze(0).to(DEVICE)
    out = model.generate(ids, max_new_tokens=max_new, temperature=0.7, top_k=40)
    return decode(out[0])


def star_finetune_step(
    model,
    rationale_pairs: List[Tuple[str, str]],
    n_epochs: int = 2,
    lr: float = 5e-5
) -> NanoGPT:
    """Fine-tune the model on the bootstrapped (question, rationale) pairs."""
    texts  = [f'{q}\n{r}' for q, r in rationale_pairs]
    ds     = CoTDataset(texts, tokenizer, loss_on_answer_only=False)
    if len(ds) == 0:
        return model
    loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=CoTDataset.collate_fn)
    opt    = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()
    for _ in range(n_epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            _, loss = model(x, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    return model


def run_star(
    base_model: NanoGPT,
    questions_and_answers: List[Tuple[str, int]],
    star_rounds: int = 3,
    K: int = 4
) -> NanoGPT:
    """
    Full STaR training loop.
    `questions_and_answers`: list of (question_string, correct_int_answer)
    """
    model = copy.deepcopy(base_model)

    for rnd in range(1, star_rounds + 1):
        collected_pairs = []
        n_self = 0
        n_hint = 0
        n_fail = 0

        print(f'  STaR Round {rnd}/{star_rounds} — generating rationales …')
        for q, ans in questions_and_answers:
            # Try to self-generate a correct rationale
            rationale = star_generate_rationale(model, q, K=K)

            if rationale is not None:
                collected_pairs.append((q, rationale))
                n_self += 1
            else:
                # Rationalise with hint
                rationale = star_hint_rationalise(model, q, ans)
                if verifier_reward(q, rationale) == 1.0:
                    collected_pairs.append((q, rationale))
                    n_hint += 1
                else:
                    n_fail += 1

        print(f'    Self-correct: {n_self} | Hint-correct: {n_hint} | Failed: {n_fail}')
        print(f'    Fine-tuning on {len(collected_pairs)} examples …')
        model = star_finetune_step(model, collected_pairs)

        # Evaluate accuracy after this round
        correct = sum(
            verifier_reward(q, cot_inference(model, q)) for q, _ in questions_and_answers
        )
        acc = correct / len(questions_and_answers)
        print(f'    Accuracy after round {rnd}: {acc*100:.1f}%\n')

    return model


# ── Run STaR ──────────────────────────────────────────────────────────────────
star_qa = [(f'What is {a} + {b}?', a + b)
           for a, b in [(random.randint(1,10), random.randint(1,10)) for _ in range(30)]]

print('Running STaR bootstrapping:')
star_model = run_star(cot_model, star_qa, star_rounds=3, K=3)

print('\nSTaR-trained model samples:')
for q, expected in random.sample(star_qa, 3):
    pred = extract_final_answer(cot_inference(star_model, q))
    print(f'  Q: {q}  →  predicted: {pred}  |  expected: {expected}')

### 2.6 Reasoning Approaches: Benchmark Comparison

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compare all 5 approaches on a held-out arithmetic test set
# ─────────────────────────────────────────────────────────────────────────────

test_qa = [(f'What is {a} + {b}?', a + b)
           for a, b in [(random.randint(1,15), random.randint(1,15)) for _ in range(40)]]


def eval_approach(model, approach_name: str, use_sc: bool = False) -> float:
    correct = 0
    for q, expected in test_qa:
        if use_sc:
            pred_str, _ = self_consistency(model, q, K=5)
        else:
            pred_str = extract_final_answer(cot_inference(model, q))
        try:
            correct += int(int(float(pred_str)) == expected)
        except (ValueError, TypeError):
            pass
    acc = correct / len(test_qa)
    print(f'  {approach_name:<40s}: {acc*100:.1f}%  ({correct}/{len(test_qa)})')
    return acc


print('=' * 60)
print('REASONING APPROACH COMPARISON (arithmetic test set)')
print('=' * 60)
results = {}
results['Base LM (greedy)']            = eval_approach(model,       'Base LM (greedy)')
results['CoT SFT (greedy)']            = eval_approach(cot_model,   'CoT SFT (greedy)')
results['CoT SFT + Self-Consistency']  = eval_approach(cot_model,   'CoT SFT + Self-Consistency (K=5)', use_sc=True)
results['PPO RL Reasoning']            = eval_approach(rl_model,    'PPO RL Reasoning')
results['STaR Bootstrapping']          = eval_approach(star_model,  'STaR Bootstrapping (3 rounds)')
print('=' * 60)

# ── Bar chart ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
names  = list(results.keys())
values = [v * 100 for v in results.values()]
colors = ['#64748b', '#3B82F6', '#06B6D4', '#EF4444', '#10B981']

bars = ax.barh(names, values, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Reasoning Accuracy by Approach\n(Arithmetic test set)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('reasoning_comparison.png', dpi=150)
plt.show()
print('Plot saved → reasoning_comparison.png')

### 2.7 Summary of Reasoning Approaches

In [ ]:
summary = {
    'Method': ['CoT SFT', 'Self-Consistency', 'PRM + Beam Search', 'PPO RLHF', 'STaR'],
    'Training required': ['Yes (supervised)', 'No (decoding only)', 'Yes (PRM)', 'Yes (RL)', 'Yes (iterative)'],
    'Extra inference cost': ['None', 'K× sampling', 'Beam × PRM calls', 'None after training', 'None after training'],
    'Key advantage': [
        'Simple & effective first step',
        'No extra training, boosts accuracy',
        'Prefers correct intermediate steps',
        'Maximises verified reward signal',
        'Self-improves using own outputs',
    ],
    'Key limitation': [
        'Requires gold CoT annotations',
        'High inference cost (K samples)',
        'Requires step-level annotations for PRM',
        'Training instability, reward hacking',
        'Slow to bootstrap; error amplification',
    ]
}

print('\nReasoning Approaches — Summary')
print('=' * 100)
row_fmt = '{:<22} {:<22} {:<22} {:<30}'
print(f'{"Method":<22} {"Training":<22} {"Inference cost":<22} {"Key advantage":<30} {"Limitation"}')
print('-' * 120)
for i in range(5):
    print(f'{summary["Method"][i]:<22}'
          f'{summary["Training required"][i]:<25}'
          f'{summary["Extra inference cost"][i]:<28}'
          f'{summary["Key advantage"][i]:<38}'
          f'{summary["Key limitation"][i]}')
print('=' * 120)

print('''
Recommendation for a small LLM (<4 GB RAM):

  Stage 1 — CoT SFT      : Always fine-tune on chain-of-thought data first. Cheap and high-impact.
  Stage 2 — Self-Consistency: Free accuracy boost at test time. Use K=5–10.
  Stage 3 — STaR          : If labelled CoT data is scarce, bootstrap rationales iteratively.
  Stage 4 — PPO/GRPO       : Apply RL with a verifier once a good base policy exists.
  Stage 5 — PRM            : Add step-level scoring for complex multi-step problems.
''')